# PySpark 12: internals and monitoring

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-12-internals-and-monitoring.ipynb)

An orders job that used to finish before the reporting deadline has slowed down. Before changing any setting,
you need to see what Spark is actually doing: the plan it chose, where it shuffles, which stage is slow, and how
memory is used on each executor.

This lesson reads physical plans and cost estimates, collects runtime evidence from the Spark UI and event logs,
and explains the machinery behind most performance problems: shuffles, adaptive query execution, caching and
checkpoints, driver and executor memory, spill and garbage collection. [PySpark 13: performance tuning](pyspark-13-performance-tuning.ipynb) turns this evidence into fixes.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 1: how Spark runs your code](pyspark-01-how-spark-runs.ipynb) and [PySpark 6: joins](pyspark-06-joins.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The session below also writes an event log, which the monitoring section reads back. The small tables from
[PySpark 1](pyspark-01-how-spark-runs.ipynb#sample-data-used-in-every-module) are too small to show a shuffle or
a straggler, so this module builds a larger pair of tables instead.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write
EVENT_LOG_DIR = os.path.join(WORK_DIR, "event-log")  # read back in the monitoring section
os.makedirs(EVENT_LOG_DIR)

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", EVENT_LOG_DIR)
    .config("spark.eventLog.compress", "false")  # plain JSON lines, readable below
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date

from pyspark.sql import functions as F

import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints, so a cell can pick out the lines that matter."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()

`big_orders` has 200,000 rows, and customer `0` owns 40 percent of them: the "guest checkout" key that makes
real joins skew. `big_customers` is a 5,000-row dimension. The cell also resets the session to a static
baseline (16 shuffle partitions, AQE off), so each lever can be switched on deliberately.

In [ ]:
# Tuning lessons start from a static baseline; later cells switch levers on one at a time.
spark.conf.set("spark.sql.shuffle.partitions", "16")
spark.conf.set("spark.sql.adaptive.enabled", "false")

from pyspark.sql.functions import spark_partition_id


def partition_sizes(df):
    """Row count per partition, in partition order: the skew signal the Stages tab shows."""
    return [r["count"] for r in df.groupBy(spark_partition_id().alias("p")).count().orderBy("p").collect()]

ROWS, CUSTOMERS, HOT_SHARE = 200_000, 5_000, 0.4
big_orders = spark.range(ROWS).select(
    F.col("id").alias("order_id"),
    F.when(F.col("id") % 10 < HOT_SHARE * 10, F.lit(0)).otherwise(F.col("id") % CUSTOMERS).alias("customer_id"),
    F.date_add(F.lit("2026-10-01").cast("date"), (F.col("id") % 30).cast("int")).alias("order_date"),
    (F.col("id") % 997 / 10).cast("decimal(12,2)").alias("amount"),
    F.element_at(F.array(F.lit("paid"), F.lit("shipped"), F.lit("placed")), (F.col("id") % 3 + 1).cast("int")).alias("status"),
)
big_customers = spark.range(CUSTOMERS).select(
    F.col("id").alias("customer_id"),
    F.concat(F.lit("segment-"), (F.col("id") % 4).cast("string")).alias("customer_segment"),
)

# A partitioned Parquet copy, so scans, pruning and pushdown behave like a real table
ORDERS_PATH = os.path.join(WORK_DIR, "big_orders")
big_orders.write.mode("overwrite").partitionBy("order_date").parquet(ORDERS_PATH)
big_orders = spark.read.parquet(ORDERS_PATH)

hot = big_orders.where("customer_id = 0").count()
print(f"rows: {big_orders.count():,}  hot-key rows: {hot:,}")

## Read a physical plan

Read `explain("formatted")` bottom-up and look for:

- `Scan parquet` with `ReadSchema`: which columns are read;
- `PartitionFilters`: directories pruned at plan time; `PushedFilters`: predicates the file reader evaluates;
- `Exchange`: a shuffle, and therefore a stage boundary;
- `BroadcastHashJoin` versus `SortMergeJoin`: whether the join shuffled both sides;
- `AdaptiveSparkPlan isFinalPlan=false`: AQE may still rewrite the plan at runtime.

Catalyst runs analysis (resolve names and types), logical optimization (pushdown, constant folding, column
pruning), physical planning (pick the cheapest candidate from statistics), and whole-stage code generation
(operators fused into one generated function, shown as `WholeStageCodegen` or `*`). `explain("cost")` prints
the size and row estimates the planner used.

**Where it matters**
- **Before tuning anything:** the plan tells you whether the problem is a scan, a shuffle or a join.
- **Code review:** an unexpected `Exchange` or a missing `PartitionFilters` is visible before the job runs.

**Watch out for**
- Treating the root of the plan as the expensive part. Cost sits in the scans, exchanges and joins.
- A pre-execution AQE plan is a guess. Confirm with the final plan or the stage metrics.

### Example

Filter by date, join to customers and aggregate by segment. The plan shows partition pruning, a broadcast join and one exchange; the cost plan shows the estimates.

In [ ]:
result = (
    big_orders
    .where(F.col("order_date") >= F.lit("2026-10-20"))
    .join(big_customers, "customer_id")
    .groupBy("customer_segment")
    .agg(F.sum("amount").alias("revenue"))
)
plan = plan_of(result)
print(plan)

In [ ]:
cost_plan = plan_of(result, "cost")
print(cost_plan[:1500])

## Monitor jobs with the Spark UI and event logs

The Spark UI lives on the driver (port 4040 while the application runs) and in the history server
afterwards. Each tab answers a question:

- **Stages:** sort by duration, open the slowest stage, compare the maximum task duration with the median, and
  read shuffle read and write and spill.
- **SQL / DataFrame:** the executed plan with row counts per operator, and whether AQE changed it.
- **Executors:** GC time, failed tasks, lost executors, and how evenly work is spread.
- **Storage:** which cached frames are materialized and how much memory they hold.

| Metric you see | Likely cause | Fix |
| --- | --- | --- |
| Max task duration much greater than the median | Skew or one oversized partition | AQE skew join, salt the hot key |
| Large shuffle read or write | A wide transformation | Broadcast the small side, pre-aggregate, filter earlier |
| Spill (memory) and spill (disk) above zero | Partition does not fit execution memory | More shuffle partitions, then more memory |
| GC time a large fraction of task time | Too many live Java objects | Smaller caches, serialized storage, smaller executors |
| Many tiny tasks, idle CPU | Over-partitioning | Fewer partitions, AQE coalescing |
| Tiny input tasks and slow listing | Small files | Compact, tune write layout |
| Executor lost or container killed | Overhead memory exhausted | Raise `spark.executor.memoryOverhead`, fewer cores per executor |
| Driver out of memory | `collect`, `toPandas` or a large broadcast | Aggregate in the cluster, lower the broadcast threshold |

This notebook turns the UI off to stay quiet in Colab, so the cells read the same evidence from plans,
partition counts and the event log. Event logs are JSON lines, one event per line; production jobs set
`spark.eventLog.enabled=true` and `spark.eventLog.dir` so the history server can show finished applications.

**Where it matters**
- **Slow-job investigations:** the stage table and the metric table above name the class of problem.
- **Regression tracking:** per-stage shuffle and spill bytes from event logs, compared run to run.

**Watch out for**
- Raising executor memory because a stage is slow. Diagnose the class first: spill, skew, shuffle size or tiny tasks each have a different lever.
- `explain` is a plan, not a runtime fact. Only the UI and the event log show what actually happened.

### Example: read stage metrics from the event log

Run one shuffle, then read this session's event log and sum shuffle and spill bytes per stage.

In [ ]:
import glob
import json
import os
from collections import defaultdict


def event_log_files(log_path):
    """Spark 3 writes one event log file; Spark 4 writes a directory of rolling event files."""
    if os.path.isdir(log_path):
        return sorted(glob.glob(os.path.join(log_path, "events_*")))
    return [log_path]


def stage_metrics(log_path):
    """Sum tasks, shuffle and spill bytes per stage from a Spark event log."""
    totals = defaultdict(lambda: {"tasks": 0, "shuffle_read": 0, "shuffle_write": 0, "spill_disk": 0, "spill_mem": 0})
    for path in event_log_files(log_path):
        with open(path) as handle:
            for line in handle:
                event = json.loads(line)
                if event.get("Event") != "SparkListenerTaskEnd":
                    continue
                metrics = event.get("Task Metrics") or {}
                read = metrics.get("Shuffle Read Metrics") or {}
                write = metrics.get("Shuffle Write Metrics") or {}
                stage = totals[event["Stage ID"]]
                stage["tasks"] += 1
                stage["shuffle_read"] += read.get("Remote Bytes Read", 0) + read.get("Local Bytes Read", 0)
                stage["shuffle_write"] += write.get("Shuffle Bytes Written", 0)
                stage["spill_disk"] += metrics.get("Disk Bytes Spilled", 0)
                stage["spill_mem"] += metrics.get("Memory Bytes Spilled", 0)
    return dict(totals)

spark.conf.set("spark.sql.adaptive.enabled", "false")
big_orders.groupBy("customer_id").agg(F.sum("amount")).collect()  # one shuffle to measure

(log_path,) = glob.glob(os.path.join(EVENT_LOG_DIR, "*"))
metrics = stage_metrics(log_path)
heaviest = max(metrics.items(), key=lambda item: item[1]["shuffle_write"])
print(f"{len(metrics)} stages in the log; heaviest shuffle writer: stage {heaviest[0]} {heaviest[1]}")

### Example: log row counts at a pipeline boundary

Row counts are the cheapest monitoring signal: a step that suddenly drops or multiplies rows is a correctness
bug before it is a performance one. Log input and output counts for a step into a metrics table.

In [ ]:
input_rows = big_orders.count()
paid = big_orders.where("status = 'paid'")
output_rows = paid.count()

spark.createDataFrame(
    [("orders_paid", input_rows, output_rows)], "step string, input_rows long, output_rows long"
).write.mode("append").saveAsTable("ops_pipeline_metrics")

logged = spark.table("ops_pipeline_metrics").first()
print(logged.asDict())

## Understand why shuffles are expensive

A shuffle redistributes rows so that all rows with the same key land in the same partition. `groupBy`,
`distinct`, `dropDuplicates`, non-broadcast joins, `repartition`, `orderBy` and windows with `partitionBy` all
shuffle. Each map task partitions its output by `hash(key) % numPartitions`, writes one block per reducer to
local disk, and reducers fetch their blocks from every mapper over the network: serialization, disk, network
and a sort or hash build.

**Where it matters**
- **Large aggregations and joins,** where shuffle bytes usually dominate runtime.
- **Unstable clusters:** a lost executor loses its shuffle files, and the map stage runs again.

**Watch out for**
- `orderBy` before a write is a range shuffle; drop it unless readers need sorted files.
- `repartition()` with no arguments is a full shuffle for no benefit.
- Moving 500 GB at an aggregate 1 GB/s costs at least 500 seconds before serialization and sorting.

### Example

Count the exchanges: a `groupBy` adds one, and an `orderBy` on top adds a range-partitioned second one.

In [ ]:
daily_revenue = big_orders.groupBy("order_date").agg(F.sum("amount").alias("revenue"))
ordered = daily_revenue.orderBy("order_date")

for name, frame in [("groupBy", daily_revenue), ("groupBy + orderBy", ordered)]:
    print(f"{name:<18} exchanges: {plan_of(frame, 'simple').count('Exchange ')}")

## Let adaptive query execution re-plan

AQE (on by default since Spark 3.2) re-plans between query stages. When the map side of a shuffle finishes,
Spark knows the real partition sizes and can:

- coalesce many small post-shuffle partitions (`spark.sql.adaptive.coalescePartitions.enabled`);
- split skewed join partitions (`spark.sql.adaptive.skewJoin.enabled`); a partition counts as skewed above both
  `skewedPartitionThresholdInBytes` (256 MB) and `skewedPartitionFactor` (5) times the median;
- switch a sort-merge join to a broadcast join when a side turns out to be small;
- read shuffle files locally (`spark.sql.adaptive.localShuffleReader.enabled`).

**Where it matters**
- **Uneven shuffles:** 190 partitions of 5 MB and ten of 2 GB become a few 64 MB partitions plus split skewed ones.
- **Filters the optimizer cannot estimate,** which leave a side far smaller than planned.

**Watch out for**
- AQE acts only at shuffle boundaries. It cannot fix a skewed scan, a bad partition key, or a scan that reads every file.
- Reading the final plan needs a completed action; the example uses an internal API (`_jdf`) that Spark Connect does not have.

### Example

The optimizer's estimate rules out a broadcast; after running, AQE has switched to one because the filtered side is tiny.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(64 * 1024))  # 64 KB: too small for the raw scan

tiny_customers = big_customers.where("customer_id < 50")  # the planner cannot know this is tiny
joined = big_orders.join(tiny_customers.groupBy("customer_id").agg(F.max("customer_segment").alias("seg")), "customer_id")
before = plan_of(joined)
joined.collect()  # run it so AQE can re-plan with real sizes
final = joined._jdf.queryExecution().executedPlan().toString()  # the plan that actually ran (internal API)
print("initial plan:", "SortMergeJoin" if "SortMergeJoin" in before else "?",
      "| final plan:", "BroadcastHashJoin" if "BroadcastHashJoin" in final else "?")

spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

## Cache, persist and checkpoint

Cache only a DataFrame that is reused, expensive to recompute, and fits. `cache()` and `persist()` are lazy:
blocks are stored on the first action. `MEMORY_ONLY` keeps deserialized objects (fast, large; evicted
partitions are recomputed), `MEMORY_AND_DISK` spills evicted blocks, and `DISK_ONLY` always writes to disk.
Storage shares the unified memory region with execution, so a large shuffle can evict cached blocks.

A **checkpoint** is different: `df.checkpoint()` writes the data to reliable storage (the directory set with
`setCheckpointDir`) and **cuts the lineage**, so later plans start from the saved data. `localCheckpoint()`
writes to executor storage instead: faster, but lost with the executor. Use them for very long plans, such
as iterative loops, where analysis time or recomputation after a failure becomes the problem.

**Where it matters**
- **Caching:** a cleaned frame read by both a valid-rows write and a quarantine write.
- **Checkpointing:** an iterative algorithm or a loop that adds columns hundreds of times.

**Watch out for**
- 100 GB cached on 20 executors with 16 GB heaps does not fit: the unified region is 0.6 of (heap - 300 MB), about 188 GB, and storage keeps only half of it safe from eviction.
- Caching by reflex, caching a frame read once, and forgetting `unpersist()`.
- A checkpoint is a full write of the data; it is not free.

### Example

Persist a filtered frame that two actions reuse, then checkpoint a frame built in a loop and check that its plan no longer reaches the Parquet scan.

In [ ]:
from pyspark import StorageLevel

base = (
    big_orders
    .where("order_date >= date '2026-10-20'")
    .select("order_id", "customer_id", "amount", "order_date")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
high_value = base.where("amount >= 90").count()  # the first action materializes the cache
daily = base.groupBy("order_date").agg(F.sum("amount").alias("revenue")).count()

print("storage level:", base.storageLevel, "| high-value big_orders:", high_value, "| days:", daily)
base.unpersist()

In [ ]:
spark.sparkContext.setCheckpointDir(os.path.join(WORK_DIR, "checkpoints"))

looped = big_orders.select("order_id", "amount")
for step in range(5):  # stands in for an iterative job whose plan keeps growing
    looped = looped.withColumn(f"amount_{step}", F.col("amount") + step)

saved = looped.checkpoint()  # eager by default: writes the data and cuts the lineage
print("--- before the checkpoint ---")
looped.explain()
print("--- after the checkpoint ---")
saved.explain()

**What to notice:** Before the checkpoint, the plan still starts from `Scan parquet` and repeats every projection. After it, the
plan starts from `Scan ExistingRDD`: the saved data. A failure or a further loop no longer replays the
lineage.

## Avoid driver memory problems

The driver holds the plan, scheduler state, broadcast values and every collected result. `collect()` and
`toPandas()` pull the whole result into one JVM; `spark.driver.maxResultSize` (1 GB) caps it, and the driver
heap often fails first. A 50-million-row result at 200 bytes per row is 10 GB. Aggregate in the cluster,
write large results to tables, and collect only bounded samples.

**Where it matters**
- **Notebooks and reports** that pull data into pandas for plotting.
- **Large broadcasts,** which are built on the driver first.

**Watch out for**
- `toPandas()` on an unfiltered table.
- Raising `maxResultSize` instead of reducing the result.

### Example

Aggregate in the cluster and bring back one row, and take a bounded sample.

In [ ]:
summary = big_orders.agg(F.sum("amount").alias("total"), F.count("*").alias("rows")).first()
sample = big_orders.select("order_id", "customer_id").limit(5).collect()

print("aggregated in the cluster:", summary.asDict())
print("bounded sample:", [row["order_id"] for row in sample])

## Size executors, memory and parallelism

Three independent numbers: cores per executor (task slots), memory per executor (shared by those tasks), and
executor count. Total parallelism is `executors x cores`, and shuffle partitions should be a small multiple of
it so the last wave does not leave cores idle.

An executor heap loses 300 MB of reserved memory; the unified region is `spark.memory.fraction` (0.6) of the
rest, shared by execution (shuffles, sorts, joins) and storage (cache), with `spark.memory.storageFraction`
(0.5) protected for storage. `spark.executor.memoryOverhead` (default `max(384 MB, 0.10 x heap)`) covers Python
workers, network buffers and JVM internals, and the container is heap plus overhead.

```python
spark = (
    SparkSession.builder
    .appName("large-orders-aggregation")
    .config("spark.executor.instances", "60")
    .config("spark.executor.cores", "5")
    .config("spark.executor.memory", "38g")
    .config("spark.executor.memoryOverhead", "4g")
    .config("spark.sql.shuffle.partitions", "1200")
    .getOrCreate()
)
```

**Where it matters**
- **Cluster sizing:** 10 nodes with 32 cores and 256 GB each; leave a core and 1 GB per node for the OS; five cores per executor gives 6 executors per node, 60 in total, about 42 GB each, roughly 38 GB of heap after overhead.
- **Shared clusters:** dynamic allocation (`spark.dynamicAllocation.*` with shuffle tracking) suits bursty jobs; static allocation suits stable nightly jobs.

**Watch out for**
- Very wide executors create long GC pauses; many one-core executors multiply JVM overhead and broadcast copies. Three to five cores and 8 to 32 GB is the usual landing zone.
- Containers killed by the cluster manager usually mean overhead memory, especially for Python-heavy jobs.
- These are launch-time settings; `spark.conf.set` cannot change them in a running session.

### Example

Compute the container size, the unified memory region and the execution memory per task for a 16 GB, five-core executor.

In [ ]:
def executor_layout(heap_gb, cores, memory_fraction=0.6, reserved_gb=0.3):
    overhead_gb = max(0.384, 0.10 * heap_gb)
    unified_gb = memory_fraction * (heap_gb - reserved_gb)
    return {
        "container_gb": round(heap_gb + overhead_gb, 1),
        "unified_gb": round(unified_gb, 1),
        "execution_per_task_gb": round(unified_gb / cores, 2),
    }

layout = executor_layout(heap_gb=16, cores=5)
print(layout)

## Handle spill, serialization and GC

When a sort, hash aggregate or hash join exhausts its task's execution memory, Spark writes sorted runs to
local disk and merges them later: correct, but disk-bound. Spill is per task, so spill on a handful of tasks
is itself a skew signal. The cause is almost always partition size versus per-task memory: a 3 GB partition
with about 1 GB per task spills.

Fix spill in this order: more shuffle partitions or a smaller advisory size; filter and aggregate earlier;
drop an unnecessary sort; broadcast the small side; only then add executor memory.

DataFrames already use Tungsten's compact binary format. Kryo serialization
(`spark.serializer=org.apache.spark.serializer.KryoSerializer`) matters for RDD shuffles, broadcasts and
caches of objects. High GC time means too many live objects: oversized caches, very wide rows or giant
executors. G1 (`-XX:+UseG1GC` in `spark.executor.extraJavaOptions`) is the usual collector.

**Where it matters**
- **Stages that spill** on large joins and aggregations.
- **RDD-heavy or cache-heavy jobs** with high GC time.

**Watch out for**
- If one partition holds 10 percent of a 2 TB shuffle, no amount of memory helps: the partition is the problem.
- Caching more while already spilling makes it worse.
- Expecting a serializer change to speed up an I/O-bound scan or a skewed task.

### Example

A stage shuffles 2 TB into 200 partitions: about 10 GB each, against roughly 1.9 GB of execution memory per
task on the executor above. Every task spills. At a 64 MB advisory size AQE targets about 32,000 partitions,
each well inside task memory, and spill drops to zero without adding memory:

```python
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
spark.conf.set("spark.sql.shuffle.partitions", "32000")  # the starting count AQE coalesces from
```

## What changes between Spark 3 and Spark 4

Most tuning advice carries over unchanged. These are the differences that show up in jobs and interviews:

- **ANSI SQL mode** is on by default in Spark 4.0 (`spark.sql.ansi.enabled=true`): invalid casts, overflow and
  division by zero raise errors instead of returning null. Jobs that silently produced nulls on Spark 3 can
  fail after an upgrade; that is a correctness fix.
- **AQE** has been on by default since 3.2. Start from AQE and tune `advisoryPartitionSizeInBytes` before
  hand-setting shuffle partitions per query.
- **Event logs** roll by default, so each application writes a directory of `events_*` files. The
  `event_log_files` helper above handles both layouts.
- **Runtime:** Spark 4 requires Java 17 or 21 and Scala 2.13. Old Java 8 GC flags do not carry over.
- **Spark Connect** is a first-class option. A remote client has no `df._jdf` or `sparkContext`, so plan
  inspection goes through `explain` and the UI.
- **Python:** PySpark 4.0 needs Python 3.9 or newer, and pandas UDFs need `pandas` and `pyarrow` on every
  worker.

Read each release's migration guide before upgrading a cluster; behaviour changes are listed there.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Where do you look in a physical plan for shuffles, pruning and the join strategy?

**A.** `Exchange` nodes are shuffles; `PartitionFilters` and `PushedFilters` on the scan show pruning and pushdown; `BroadcastHashJoin` or `SortMergeJoin` shows the join strategy.

**Q2.** Why is `explain` not enough to diagnose a slow job?

**A.** It shows the plan, not what happened. Task durations, shuffle bytes, spill and GC time come from the Spark UI or the event log.

**Q3.** One task takes 12 minutes while the median takes 20 seconds. What is it?

**A.** Skew: one partition holds far more data than the others. More executors will not help.

**Q4.** What can adaptive query execution change at runtime, and what can it not fix?

**A.** It coalesces small shuffle partitions, splits skewed join partitions and switches joins to broadcast. It acts only at shuffle boundaries, so it cannot fix a skewed scan or a scan that reads every file.

**Q5.** How is `checkpoint()` different from `cache()`?

**A.** `cache` keeps data for reuse but keeps the lineage; `checkpoint` writes the data to reliable storage and cuts the lineage.

**Q6.** A container is killed by the cluster manager. Which memory setting do you check first?

**A.** `spark.executor.memoryOverhead`, which covers Python workers and off-heap memory on top of the JVM heap.

## Key takeaways

- Read the plan first: scans, exchanges, join strategies and pushed filters.
- Runtime facts come from the UI and event logs: task time distribution, shuffle bytes, spill, GC.
- Shuffles are the main cost; AQE improves them at shuffle boundaries only.
- Cache only what several actions reuse; checkpoint to cut long lineages.
- Executor memory is heap plus overhead; the unified region is shared by execution and cache.

**Next:** Continue with [PySpark 13: performance tuning](pyspark-13-performance-tuning.ipynb).